In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [8]:
pd.set_option('display.max_columns',None)
url = 'https://raw.githubusercontent.com/ichiP245/my-next-soderling/refs/heads/main/Archivos/pre_ml_df.csv'

df = pd.read_csv(url)
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

## Encoding de categóricas

Como el objetivo es que estos datos entren a un modelo de Machine Learning, le hacemos un encoding a las variables categóricas.

Para eso exploramos un poco esas variables. Sin embargo, para análisis exploratorio multivariable usaremos el df que acabamos de guardar, ya que tiene estas variables no encodeadas y eso nos va a facilitar el análisis para tal caso.

Para el nivel del torneo podemos hacer un OrdinalEncoder tranquilamente

In [9]:
df['Series'].value_counts()

,count
Series,
Masters 1000,5884
Grand Slam,5459
ATP500,3954
ATP250,764


In [10]:
from sklearn.preprocessing import OrdinalEncoder # Ordinal Encoder -> con valores de 0 a n cantidad de variables

oe = OrdinalEncoder(categories=[['ATP250', 'ATP500', 'Masters 1000', 'Grand Slam']])
series_oe = oe.fit_transform(df[['Series']])
series_oe = pd.DataFrame(series_oe, columns=['series_level_oe'])
df = pd.concat([df, series_oe], axis=1)

Para Round armamos varias One Hot, porque tenemos ['1st Round', '2nd Round', 'Quarterfinals', 'Semifinals', 'The Final', '3rd Round', '4th Round'] y no sirve hacer un solo OneHot o un LabelEncoder ni otro.

Entonces encodeamos las instancias que son mas importantes: si es o no una final, si es o no una semifinal (donde puede llegar alguno que hizo un batacazo y eso puede influenciar) y si es o no 1st round

In [11]:
df['Round'].value_counts()

,count
Round,
1st Round,7252
2nd Round,4541
3rd Round,1768
Quarterfinals,1096
4th Round,583
Semifinals,548
The Final,273


In [12]:
round_order = {
    '1st Round': 1, '2nd Round': 2, '3rd Round': 3,
    '4th Round': 4, 'Quarterfinals': 5, 'Semifinals': 6, 'The Final': 7
}

df['round_encoded'] = df['Round'].map(round_order)

"Best of" es numerica pero categorica. Le hacemos One Hot

In [13]:
df['Best of'].value_counts()

,count
Best of,
3.0,10612
5.0,5449


In [14]:
df['is_best_of_5'] = (df['Best of'] == 5).astype(int)

Para Surface = ['Hard', 'Clay', 'Grass'] usamos FrequencyEncoder

In [15]:
df['Surface'].value_counts(normalize=True)

,proportion
Surface,
Hard,0.608617
Clay,0.293008
Grass,0.098375


In [16]:
surface_fe = df['Surface'].value_counts(normalize=True) # Generamos un pd.Series con la proporcion de cada valor unicos
df['surface_fe'] = df['Surface'].map(surface_fe)  # Armamos una variable donde cada valor recibe la frecuencia de aparicion de ese valor unico

Para Court = ['Outdoor', 'Indoor'] usamos OneHotEncoder

In [17]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first')
court_encoded = ohe.fit_transform(df[['Court']])
court_encoded = pd.DataFrame(court_encoded, columns=ohe.get_feature_names_out()).rename(columns={'Court_Outdoor':'is_outdoor'})
df = pd.concat([df, court_encoded], axis=1)

Volvemos a guardar el df

In [19]:
columns_drop = ['Series', 'Court', 'Surface', 'Round', 'Best of']

In [20]:
df = df.drop(columns=columns_drop)

## Sigue preparación para ML

In [21]:
new_df = df.drop(columns=['Location', 'Fecha', 'playerA', 'playerB',
                          'A1', 'B1', 'A2', 'B2', 'A3', 'B3', 'A4', 'B4', 'A5',
                          'B5', 'setsA', 'setsB', 'setsPartido'])

In [23]:
df_no_nulls = new_df.dropna()

In [24]:
X = df_no_nulls.drop(columns=['target'])
y = df_no_nulls['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

## Modelo 0: baseline

In [25]:
y_train_baseline = X_train[['rankA', 'rankB']].apply(lambda row: 1 if row['rankA'] > row['rankB'] else 0, axis=1)
y_test_baseline = X_test[['rankA', 'rankB']].apply(lambda row: 1 if row['rankA'] > row['rankB'] else 0, axis=1)

In [31]:
print(confusion_matrix(y_train, y_train_baseline))

[[2130 4113]
 [4161 2008]]


In [27]:
print(classification_report(y_train, y_train_baseline))

              precision    recall  f1-score   support

           0       0.34      0.34      0.34      6243
           1       0.33      0.33      0.33      6169

    accuracy                           0.33     12412
   macro avg       0.33      0.33      0.33     12412
weighted avg       0.33      0.33      0.33     12412



In [30]:
print(confusion_matrix(y_test, y_test_baseline))

[[ 530 1063]
 [1026  485]]


In [ ]:
print(classification_report(y_test, y_test_baseline))

              precision    recall  f1-score   support

           0       0.34      0.33      0.34      1593
           1       0.31      0.32      0.32      1511

    accuracy                           0.33      3104
   macro avg       0.33      0.33      0.33      3104
weighted avg       0.33      0.33      0.33      3104



Logistic

In [97]:
X_train.shape

(12412, 72)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_grid = {
    'logistic__C'          : np.logspace(-3, 3, 20),  # Regularización
    'logistic__penalty'    : ['l1', 'l2'],
    'logistic__solver'     : ['liblinear'],
}

pipeline = Pipeline([
    ('scaler',   StandardScaler()),
    ('logistic', LogisticRegression(max_iter=1000, random_state=42))
])

search_lr = RandomizedSearchCV(
    estimator           = pipeline,
    param_distributions = param_grid,
    n_iter              = 30,
    cv                  = 5,
    scoring             = 'roc_auc',
    n_jobs              = 1,
    random_state        = 42,
    verbose             = 2
)

search_lr.fit(X_train, y_train)

print(f"Mejores parámetros: {search_lr.best_params_}")
print(f"Mejor ROC-AUC: {search_lr.best_score_:.4f}")

modelo_lr = search_lr.best_estimator_

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END logistic__C=0.6951927961775606, logistic__penalty=l2, logistic__solver=liblinear; total time=   1.1s
[CV] END logistic__C=0.6951927961775606, logistic__penalty=l2, logistic__solver=liblinear; total time=   0.7s
[CV] END logistic__C=0.6951927961775606, logistic__penalty=l2, logistic__solver=liblinear; total time=   0.7s
[CV] END logistic__C=0.6951927961775606, logistic__penalty=l2, logistic__solver=liblinear; total time=   0.6s
[CV] END logistic__C=0.6951927961775606, logistic__penalty=l2, logistic__solver=liblinear; total time=   1.1s
[CV] END logistic__C=0.3359818286283781, logistic__penalty=l1, logistic__solver=liblinear; total time=   2.4s
[CV] END logistic__C=0.3359818286283781, logistic__penalty=l1, logistic__solver=liblinear; total time=   0.7s
[CV] END logistic__C=0.3359818286283781, logistic__penalty=l1, logistic__solver=liblinear; total time=   0.7s
[CV] END logistic__C=0.3359818286283781, logistic__penalty

In [1]:
y_pred_lr      = modelo_lr.predict(X_test)
y_pred_lr_prob = modelo_lr.predict_proba(X_test)[:, 1]

NameError: name 'modelo_lr' is not defined

Modelo 1: RandomForest

In [32]:
rnd_clf = RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_split=4, random_state=42)
rnd_clf.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, min_samples_split=4, n_estimators=500,
                       random_state=42)

In [33]:
y_pred_rf_train = rnd_clf.predict(X_train)
y_pred_rf_test = rnd_clf.predict(X_test)

In [34]:
print('Matriz de confusión de datos de train:')
print(confusion_matrix(y_train, y_pred_rf_train))

Matriz de confusión de datos de train:
[[6011  232]
 [ 430 5739]]


In [35]:
print('Classification report de datos de train:')
print(classification_report(y_train, y_pred_rf_train))

Classification report de datos de train:
              precision    recall  f1-score   support

           0       0.93      0.96      0.95      6243
           1       0.96      0.93      0.95      6169

    accuracy                           0.95     12412
   macro avg       0.95      0.95      0.95     12412
weighted avg       0.95      0.95      0.95     12412



In [36]:
print('Matriz de confusión de datos de test:')
print(confusion_matrix(y_test, y_pred_rf_test))

Matriz de confusión de datos de test:
[[1127  466]
 [ 443 1068]]


In [37]:
print('Classification report de datos de test:')
print(classification_report(y_test, y_pred_rf_test))

Classification report de datos de test:
              precision    recall  f1-score   support

           0       0.72      0.71      0.71      1593
           1       0.70      0.71      0.70      1511

    accuracy                           0.71      3104
   macro avg       0.71      0.71      0.71      3104
weighted avg       0.71      0.71      0.71      3104



In [38]:
importances = rnd_clf.feature_importances_
features = X.columns
feature_importances = pd.DataFrame({'feature': features, 'importance': importances}).sort_values('importance', ascending=False)
feature_importances

,feature,importance
16,ProbMaxB,0.038533
15,ProbMaxA,0.034587
21,AvgOddsDiff,0.033169
7,MaxB,0.031642
22,logit_oddsA,0.030561
...,...,...
67,series_level_oe,0.003120
34,racha_victorias_A,0.002928
70,surface_fe,0.002208
69,is_best_of_5,0.001169


In [39]:
rnd_clf_2 = RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_split=10, random_state=42)
rnd_clf_2.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_split=10, n_estimators=500,
                       random_state=42)

In [40]:
y_pred_rf_2_train = rnd_clf.predict(X_train)
y_pred_rf_2_test = rnd_clf.predict(X_test)

In [93]:
df_testeo = pd.concat([X_test.reset_index(drop=True), pd.DataFrame(y_test).reset_index(drop=True), pd.DataFrame(y_pred_rf_test, columns=['Pred'])], axis=1)

In [94]:
df_testeo

,rankA,rankB,PtsA,PtsB,B365A,B365B,MaxA,MaxB,AvgA,AvgB,B365BookmakersMargin,B365ProbA,B365ProbB,ProbAvgA,ProbAvgB,ProbMaxA,ProbMaxB,market_uncertainty,OddsDiffB365,OddsDiffAvg,OddsDiffMax,AvgOddsDiff,logit_oddsA,logit_oddsB,rankDiff,ptsDiff,winrate_5_A,winrate_10_A,wins_before_A,win_pct_before_A,sets_5d_A,sets_30d_A,partidos_365d_A,dias_ultimo_partido_A,racha_victorias_A,winrate_5_B,winrate_10_B,wins_before_B,win_pct_before_B,sets_5d_B,sets_30d_B,partidos_365d_B,dias_ultimo_partido_B,racha_victorias_B,diff_winrate_5,diff_winrate_10,diff_win_pct_before,diff_sets_5d,diff_sets_30d,diff_dias_ultimo_partido,diff_partidos_365d,h2h_matches_previous,temperature_2m_mean,apparent_temperature_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,wind_gusts_10m_mean,wind_speed_10m_mean,relative_humidity_2m_mean,relative_humidity_2m_max,relative_humidity_2m_min,soil_temperature_0_to_100cm_mean,soil_moisture_0_to_100cm_mean,wind_gusts_10m_min,wind_speed_10m_min,series_level_oe,round_encoded,is_best_of_5,surface_fe,is_outdoor,target,Pred
0,145.0,6.0,395.0,4415.0,13.00,1.04,17.50,1.05,12.03,1.04,0.038462,0.074074,0.925926,0.079572,0.920428,0.056604,0.943396,0.420428,11.96,10.99,16.45,-0.840857,-2.448183,2.448183,139.0,-4020.0,0.2,0.3,12.0,0.400000,3.0,5.0,19,1.0,1,0.8,0.8,93.0,0.738095,0.0,10.0,49,7.0,0,-0.6,-0.5,-0.338095,3.0,-5.0,-6.0,-30,0,27.549002,24.161692,0.0,0.0,11.885453,24.840000,16.695000,7.864341,13.760123,18.907576,8.882428,20.728897,0.063580,3.240000,0.804984,2.0,2,0,0.608617,1.0,0,0
1,40.0,219.0,1070.0,256.0,1.07,9.00,1.11,9.70,1.08,8.01,0.045691,0.893744,0.106256,0.881188,0.118812,0.897317,0.102683,0.381188,-7.93,-6.93,-8.59,0.762376,2.003730,-2.003730,-179.0,814.0,0.6,0.7,118.0,0.655556,0.0,10.0,20,10.0,0,0.0,0.0,0.0,0.000000,0.0,0.0,0,1092.0,0,0.6,0.7,0.655556,0.0,10.0,-1082.0,20,0,15.797916,13.077911,0.2,0.2,27.064720,47.519997,30.210001,17.128454,65.530620,93.210000,43.928000,16.831917,0.213480,10.799999,6.696387,3.0,1,1,0.098375,1.0,1,1
2,36.0,5.0,1230.0,5085.0,2.50,1.50,2.95,1.54,2.69,1.46,0.066667,0.375000,0.625000,0.351807,0.648193,0.342984,0.657016,0.148193,1.00,1.23,1.41,-0.296386,-0.611105,0.611105,31.0,-3855.0,0.4,0.5,88.0,0.628571,0.0,0.0,21,56.0,0,0.8,0.7,167.0,0.660079,0.0,18.0,53,11.0,0,-0.4,-0.2,-0.031508,0.0,-18.0,45.0,-32,1,14.268748,14.016555,0.0,0.0,13.044722,25.199999,14.730000,6.870786,88.654260,96.439354,72.832450,14.203584,0.268558,4.680000,1.080000,1.0,1,0,0.608617,0.0,0,0
3,115.0,163.0,621.0,392.0,1.61,2.30,1.62,2.63,1.56,2.40,0.055901,0.588235,0.411765,0.606061,0.393939,0.618824,0.381176,0.106061,-0.69,-0.84,-1.01,0.212121,0.430783,-0.430783,-48.0,229.0,0.6,0.5,3.0,0.500000,3.0,11.0,3,2.0,1,0.4,0.5,3.0,0.500000,3.0,7.0,4,2.0,1,0.2,0.0,0.000000,0.0,4.0,0.0,-1,0,13.762501,9.761560,0.0,0.0,29.462870,53.639996,46.230000,25.213821,72.721504,85.680489,55.410130,10.026959,0.282667,36.360001,20.304129,0.0,2,0,0.608617,0.0,0,1
4,80.0,43.0,644.0,954.0,1.40,2.75,1.54,3.00,1.44,2.74,0.077922,0.662651,0.337349,0.655502,0.344498,0.660793,0.339207,0.155502,-1.35,-1.30,-1.46,0.311005,0.643315,-0.643315,37.0,-310.0,0.4,0.4,4.0,0.400000,0.0,0.0,8,33.0,0,0.2,0.5,11.0,0.550000,0.0,2.0,19,7.0,0,0.2,-0.1,-0.150000,0.0,-2.0,26.0,-11,0,14.640916,13.591114,0.0,0.0,17.026896,35.280000,21.255000,10.103543,76.119910,97.374700,48.936035,17.006685,0.243576,11.159999,4.334974,1.0,1,0,0.293008,1.0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3099,11.0,28.0,2900.0,1400.0,1.62,2.30,1.65,2.65,1.59,2.36,0.052067,0.586735,0.413265,0.597468,0.402532,0.616279,0.383721,0.097468,-0.68,-0.77,-1.00,0.194937,0.394928,-0.394928,-17.0,1500.0,0.8,0.9,112.0,0.592593,7.0,25.0,49,1.0,3,0.8,0.8,43.0,0.443299,8.0,27.0,31,1.0,3,0.0,0.1,0.149294,-1.0,-2.0,0.0,18,1,12.212501,9.915957

In [95]:
sum(df_testeo.apply(lambda row: row['B365A']*1 - 1 if row['target'] == row['Pred'] else -1, axis=1))

3964.646

In [96]:
def ganancia(row):
    if row['Pred'] == row['target']:  # Acertaste
        if row['Pred'] == 1:          # Apostaste a A
            return row['B365A'] - 1
        else:                          # Apostaste a B
            return row['B365B'] - 1
    else:                              # Fallaste
        return -1

df_testeo['ganancia'] = df_testeo.apply(ganancia, axis=1)
print(df_testeo['ganancia'].sum())

-116.449


Aparte:

In [75]:
from lightgbm import LGBMClassifier

modelo = LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
modelo.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 6169, number of negative: 6243
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032646 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12497
[LightGBM] [Info] Number of data points in the train set: 12412, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.497019 -> initscore=-0.011924
[LightGBM] [Info] Start training from score -0.011924


LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=42)

In [77]:
new_y_pred = modelo.predict(X_test)

In [78]:
new_y_pred

array([0, 1, 0, ..., 0, 1, 0])

In [81]:
confusion_matrix(y_test, new_y_pred)

array([[1105,  488],
       [ 465, 1046]])

In [82]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators'      : [100, 200, 500],
    'max_depth'         : [5, 10, 20, 25],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
    'max_features'      : ['sqrt', 'log2'],
}

search = RandomizedSearchCV(
    estimator   = RandomForestClassifier(random_state=42),
    param_distributions = param_grid,
    n_iter      = 50,        # Prueba 50 combinaciones aleatorias
    cv          = 5,         # 5-fold cross validation
    scoring     = 'roc_auc', # Métrica a optimizar
    n_jobs      = -1,        # Usa todos los cores
    random_state= 42
)

search.fit(X_train, y_train)

print(f"Mejores parámetros: {search.best_params_}")
print(f"Mejor ROC-AUC: {search.best_score_:.4f}")

modelo_optimo = search.best_estimator_

Mejores parámetros: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 5}
Mejor ROC-AUC: 0.7788


In [85]:
y_best_pred = modelo_optimo.predict(X_test)

In [86]:
print(confusion_matrix(y_test, y_best_pred))

[[1128  465]
 [ 416 1095]]


In [90]:
print(classification_report(y_test, y_best_pred))

              precision    recall  f1-score   support

           0       0.73      0.71      0.72      1593
           1       0.70      0.72      0.71      1511

    accuracy                           0.72      3104
   macro avg       0.72      0.72      0.72      3104
weighted avg       0.72      0.72      0.72      3104



In [91]:
# Unimos todo en un solo df para trabajar cómodo
resultados = X_test[['B365A', 'B365B']].copy()
resultados['y_real']      = y_test.values
resultados['y_pred']      = y_best_pred  # el 0/1 que devuelve el modelo
resultados['prob_mercado_A'] = X_test['B365ProbA']

In [92]:
# Cuota a cobrar según la predicción
resultados['cuota_apostada'] = resultados.apply(
    lambda row: row['B365A'] if row['y_pred'] == 1 else row['B365B'], axis=1
)

# Ganó la apuesta si la predicción coincide con el resultado real
resultados['acierto'] = resultados['y_pred'] == resultados['y_real']

# Ganancia por partido (apuesta de 1 unidad)
resultados['ganancia'] = resultados.apply(
    lambda row: row['cuota_apostada'] - 1 if row['acierto'] else -1, axis=1
)

# Resumen
n_apuestas = len(resultados)
ganancia_total = resultados['ganancia'].sum()
roi = ganancia_total / n_apuestas * 100

print(f"Apuestas realizadas: {n_apuestas}")
print(f"Aciertos: {resultados['acierto'].sum()} ({resultados['acierto'].mean()*100:.1f}%)")
print(f"Ganancia total: {ganancia_total:.2f} unidades")
print(f"ROI: {roi:.2f}%")

Apuestas realizadas: 3104
Aciertos: 2223 (71.6%)
Ganancia total: -102.46 unidades
ROI: -3.30%


In [ ]:
# Unimos todo en un solo df para trabajar cómodo
resultados = X_test[['MaxA', 'MaxB']].copy()
resultados['y_real']      = y_test.values
resultados['y_pred']      = y_best_pred  # el 0/1 que devuelve el modelo
resultados['prob_mercado_A'] = X_test['B365ProbA']